In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from load_data import load_test_rul
from evaluation import CUTOFFS, mae, rmse

MODELS = ["ols_v1", "elastic_net_v2", "random_forest", "gradient_boosting"]
THRESHOLD = 125

## 1. Load the answers and the predictions

In [ ]:
# Validation: 100 fixed cutoff points merged with all 4 model predictions
val = (
    CUTOFFS
    .merge(pd.read_csv("../results/val_predictions_linear.csv"), on=["engine_id", "cycle"])
    .merge(pd.read_csv("../results/val_predictions_trees.csv"),  on=["engine_id", "cycle"])
)

# Test: true RUL per engine merged with all 4 model predictions
test = (
    load_test_rul()
    .merge(pd.read_csv("../results/test_predictions_linear.csv"), on="engine_id")
    .merge(pd.read_csv("../results/test_predictions_trees.csv"),  on="engine_id")
)

print("val shape :", val.shape,  " — expect (100, 6)")
print("test shape:", test.shape, " — expect (100, 6)")
val.head()

## 2. `score_by_band` — split error into early and late points

**Early point:** true RUL > 125 (engine still far from failure)  
**Late point:** true RUL ≤ 125 (engine close to failure)  

Points are grouped by **true** RUL, never by the prediction — grouping by prediction would let the model choose its own group.

In [ ]:
def score_by_band(y_true, y_pred):
    """Return early/late MAE and RMSE split by THRESHOLD on the true RUL.
    
    Parameters
    ----------
    y_true : array-like  True RUL values.
    y_pred : array-like  Predicted RUL values.
    
    Returns
    -------
    dict with keys 'early' and 'late', each containing n, mae, rmse.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    early_mask = y_true > THRESHOLD
    late_mask  = ~early_mask

    result = {}
    for name, mask in [("early", early_mask), ("late", late_mask)]:
        n = int(mask.sum())
        if n == 0:
            result[name] = {"n": 0, "mae": np.nan, "rmse": np.nan}
        else:
            result[name] = {
                "n":    n,
                "mae":  mae(y_true[mask],  y_pred[mask]),
                "rmse": rmse(y_true[mask], y_pred[mask]),
            }
    return result


# Quick sanity check on the validation set
check = score_by_band(val["rul"], val["ols_v1"])
print("early n:", check["early"]["n"], "  late n:", check["late"]["n"])